# 👩‍🎓 Child Speech Recognition for Early-Grade Reading Assessments
_A PhD Theory + Lab Notebook with Presentation, Diagrams, Results & Code_

**Robert-Florin Simion | UPB ETTI**
Supervisor: Prof. Dr. Ing. Dragoș Burileanu

> 📋 Welcome! This notebook merges all theory, presentation, experiments, code labs, and charts with design, emojis, tables, and explanations. Use it for your documentation.


## 📊 Motivation & Problem
- **Existing ASR models:** Optimized for adults
- **Children's speech characteristics:** Higher pitch, less fluency
- **Goal:** Automated reading assessment for education
- **Gap:** WER on child speech ~12–15% vs <3% for adults
---

## 🎯 2. Thesis Objectives and Approach

- **General Objective**
    - Develop robust child ASR systems for educational purposes
- **Specific Objectives**
    - Match adult-level accuracy
    - Adapt ASR for phonics, decoding, syllable tasks
    - Automate scoring for reading assessments
- **Expected Deliverables**:
    - Web/mobile demo app
    - Publications: Interspeech, ICASSP...

## 🧪 3. Research Workflow & Methodology

```mermaid graph TD;
    A[Literature Review] --> B[Model Development: wav2vec2 & Whisper];
    B --> C[Data Preparation: MyST, PFSTAR, etc];
    C --> D[Experimentation: Pretrain & Finetune];
    D --> E[Evaluation: WER & Cross-validation];
    E --> F[Deployment: Apps, Papers];
```

Timeline:
- Year 1: Literature, datasets, baseline
- Year 2: Advanced models
- Year 3: Romanian corpus 
- Year 4: Deploy, write-up, publish
---

## 📚 4. Literature Review – Key Papers
### Jain et al. (2023a, IEEE Access)
• **wav2vec2**: Self-supervised, 7.42% WER (MyST)
### Jain et al. (2023b, Interspeech)
• **Whisper**: Weakly-supervised, 12.28% WER (MyST), better zero-shot
### Jain et al. (2024, IEEE Access)
• **Whisper**: Non-native scenarios, 29-89% WER improvement

---

## 🏗️ 5. Architectures

*Generics*
**🗣️ What is ASR? (Automatic Speech Recognition)**
Automatic Speech Recognition (ASR) systems convert spoken language (audio) into written text. They power virtual assistants, voice search, and transcription.

***Pipeline***
- Audio Input: Microphone or audio file
- Feature Extraction: Converts audio waveform into numerical features (MFCCs, Mel spectrogram, or neural features)
- Acoustic Model: Neural network predicts speech sounds (phonemes, graphemes)
- Language Model: Optional, predicts word sequences for context
- Decoding: Finds the most probable transcription (CTC, beam search)
- Text Output: Final transcript
![asr_flowchart.png](charts/asr_flowchart.png)

**A. wav2vec2**
wav2vec2 is a neural network architecture for automatic speech recognition, developed by Facebook AI. It uses self-supervised learning—learning from stuff without labels!
Key Components:
- Feature Extractor: 7-layer CNN learns basic audio features from the raw waveform.
- Context Network (Transformer): Learns long-range relationships in audio.
- Quantization: Discretizes the audio features (using "codebooks").
- Pretraining: Learns to predict masked audio parts using contrastive loss, so model discovers good representations by itself.
- Finetuning: Adapts with labeled speech for transcription using CTC (Connectionist Temporal Classification).
---
**Wav2Vec2(Essentials):**
- CNN feature extractor, 7 blocks
- Transformer BASE: 12 layers, 768-dim
- Gumbel softmax quantization
![wav2vec2_architecture.png](charts/wav2vec2_architecture.png)
---
**B. Whisper**
- Encoder-decoder transformer, 2 CNN layers
- 80-channel log-Mel spectrogram
- Tiny-Medium-Large models
![whisper_architecture.png](charts/whisper_architecture.png)
---

## 📊 6. Performance Benchmarks

| Dataset | wav2vec2 | Whisper |
|---------|----------|---------|
| MyST | 7.42% | 12.28% |
| PFSTAR | 2.91% | 2.98% |
| CMU-Kids | 12.77% | 15.08% |

![chart.png](charts/chart.png)
---

In [4]:
## 🛠️ 7. Laboratory Examples: Code & Experiments
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

import sys
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import librosa
import torch
import whisper

print(f"Python: {sys.executable}")
print(f"Torch version: {torch.__version__}")
# !{sys.executable} -m pip uninstall -y torch torchvision torchaudio
# !{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !{sys.executable} -m pip install transformers librosa torch soundfile openai-whisper jiwer
# print(f"Python executable: {sys.executable}")
# print(f"Python version: {sys.version}")
#### wav2vec2 Inference

model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
audio, _ = librosa.load('child.wav', sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
with torch.no_grad(): logits = model(inputs.input_values).logits
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]
print(transcription)



Python: E:\PHD\IA3\prezentare\demo-sr-ia3\.venv\Scripts\python.exe
Torch version: 2.9.0+cpu


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SHE JUMPED ROPE



## 🧑‍🏫 8. Results Table & Error Analysis

| Metric | MyST | PFSTAR | CMU-Kids |
|--------|------|--------|----------|
| wav2vec2 | 7.42 | 2.91 | 12.77 |
| Whisper | 12.28 | 2.98 | 15.08 |

### Word Error Rate (WER) Formula

$$WER = \frac{S + D + I}{N} \times 100$$

**Components:**
- **S** = Substitutions (words replaced incorrectly)
- **D** = Deletions (words missing from transcription)
- **I** = Insertions (extra words added incorrectly)
- **N** = Total number of words in reference (ground truth)

**Example:**

Reference: "the cat sat on the mat"  
Hypothesis: "the dog sat the mat"

- S = 1 (cat → dog)
- D = 1 (missing "on")
- I = 0 (no extra words)
- N = 6 (total words)

$$WER = \frac{1 + 1 + 0}{6} \times 100 = 33.3\%$$

**Interpretation:**
- Lower is better (0% = perfect)
- Adult ASR: ~2-3% WER
- Child ASR: ~12-15% WER ← Research gap

---


## 📈 9. Research Methodology & Timeline
![research_workflow.png](charts/research_workflow.png)
---

## 🚀 10. Roadmap, Next Steps & Future Work

- Short-term: dataset curation, initial training
- Medium: architecure/models adaptation
- Long-term: deployment, educational pilots, publications
- Long-long term: Romanian corpus, transfer learning from English corpus

---
## 📝 Notes & ToDo
- [ ] Atleast start :)
- [ ] Deep Learning from scratch
- [ ] Understand speech techniques under transfer learning from images
- [ ] State-of-the-art ideas
- [ ] State-of-the-art implementation
- [ ] Start with English
- [ ] Fine tuning
- [ ] Transfer to Romanian if possible (NTH)
- [ ] Deploy integrated model for testing 


---
## 📞 Contact & Utils
https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.81908&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false

https://huggingface.co/docs/transformers/model_doc/wav2vec2

wav2vec2 - https://arxiv.org/pdf/2006.11477

whisper - https://huggingface.co/spaces/openai/whisper

3blue1brown - https://www.3blue1brown.com/?v=neural-networks

Robert-Florin Simion | simion.robert.florin@gmail.com | UPB ETTI 
| Advisor: Prof. Dr. Ing. Dragoș Burileanu | robert-florin.simion@ing.com
